# Phase 2 — Delivered-Order Negative Review Classification

This notebook asks: **for any completed delivered order, can Logistic Regression predict whether the customer will subsequently give a negative review (1–2 stars)?**

This is a separate business formulation from the late-order-only notebook. It expands the population rather than replacing that analysis.

## 1. Define the problem

**Target**

* `1`: review score 1–2 (negative review)
* `0`: review score 3–5 (not negative)

**Population:** orders with delivered status, an actual customer-delivery date, and a subsequently recorded review. One row represents one eligible delivered order; late, on-time and early orders are included.

**Prediction point:** **immediately after delivery is completed, before the customer submits their review.** Delivery outcomes such as `delivery_days`, `days_early` and `late_delivery_flag` are therefore available and are not leakage. Review identifiers, text and timestamps remain excluded.

Because the target is observed only for reviewed orders, the model formally estimates negative-review risk conditional on a review being recorded. The population audit reports review coverage so this limitation is visible.

### Load the Phase 1 analytical data

The notebook uses `data/processed/order_level.csv` and `data/processed/item_level.csv`. Reusable validation and feature construction remain in `src/`.

In [ ]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.classification_data import (
    DELIVERED_CATEGORICAL_FEATURES,
    DELIVERED_MODEL_FEATURES,
    DELIVERED_NUMERIC_FEATURES,
    TARGET_COLUMN,
    build_delivered_order_classification_dataset,
    split_delivered_order_features_target,
)
from src.classification_evaluation import classification_metrics
from src.classification_model import (
    build_delivered_order_logistic_pipeline,
    predict_from_delivered_order_artifact,
    save_delivered_order_inference_artifact,
)

RANDOM_SEED = 42
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

order_level = pd.read_csv(PROCESSED_DIR / "order_level.csv")
item_level = pd.read_csv(PROCESSED_DIR / "item_level.csv")

print(f"order_level: {order_level.shape}")
print(f"item_level: {item_level.shape}")

### Build the modelling dataset

`build_delivered_order_classification_dataset(...)` validates schemas and keys, applies the delivered/date/review eligibility conditions, aggregates item features to order level, and constructs the binary target. The detailed implementation remains in `src/classification_data.py`.

The feature contract reuses the established order, payment, product, geography and timing predictors. `late_delivery_flag` becomes a predictor because lateness varies in this population. `estimated_delivery_days` is omitted because it is algebraically represented by `delivery_days` and `days_early`; avoiding all three improves coefficient interpretability.

In [ ]:
delivered_at = pd.to_datetime(
    order_level["order_delivered_customer_date"], errors="coerce"
)
population_masks = [
    ("All source orders", pd.Series(True, index=order_level.index)),
    ("Delivered status", order_level["order_status"].eq("delivered")),
    ("Actual customer-delivery date present", delivered_at.notna()),
    ("Review subsequently recorded", order_level["review_score"].notna()),
]
eligible_so_far = pd.Series(True, index=order_level.index)
population_audit_rows = []
for stage, stage_mask in population_masks:
    eligible_so_far &= stage_mask
    population_audit_rows.append({
        "filtering_stage": stage,
        "orders_remaining": int(eligible_so_far.sum()),
    })

display(pd.DataFrame(population_audit_rows))
print(f"Review coverage: {eligible_so_far.sum() / (order_level['order_status'].eq('delivered') & delivered_at.notna()).sum():.2%}")

classification_data = build_delivered_order_classification_dataset(
    order_level, item_level
)
print(f"Rows available for modelling: {len(classification_data):,}")
display(
    classification_data[TARGET_COLUMN]
    .value_counts()
    .rename(index={0: "Review 3–5", 1: "Negative review 1–2"})
    .rename("orders")
    .to_frame()
)
print(
    f"Original features: {len(DELIVERED_NUMERIC_FEATURES)} numeric and "
    f"{len(DELIVERED_CATEGORICAL_FEATURES)} categorical"
)
display(pd.DataFrame({
    "feature": DELIVERED_MODEL_FEATURES,
    "type": [
        "numeric" if name in DELIVERED_NUMERIC_FEATURES else "categorical"
        for name in DELIVERED_MODEL_FEATURES
    ],
}))

assert classification_data["order_id"].is_unique
assert "order_id" not in DELIVERED_MODEL_FEATURES
assert not any(name.startswith("review_") for name in DELIVERED_MODEL_FEATURES)

## 2. Split the data

`split_delivered_order_features_target(...)` returns `X` with only the imported predictor contract and `y` with `is_negative_review`. Identifiers and review-derived fields are excluded.

The split uses 80% for training and reserves 20% as a held-out test set. `stratify=y` preserves the comparatively small negative-review proportion in both sets.

In [ ]:
X, y = split_delivered_order_features_target(classification_data)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_SEED,
)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Negative-review rate in training: {y_train.mean():.2%}")
print(f"Negative-review rate in test: {y_test.mean():.2%}")

## 3. Train Logistic Regression

`build_delivered_order_logistic_pipeline(...)` keeps preprocessing and Logistic Regression together. Numeric features use median imputation followed by `StandardScaler`; categorical features use most-frequent imputation followed by `OneHotEncoder(handle_unknown="ignore")`. Preprocessing is therefore fitted only on the relevant training rows or folds.

Five-fold stratified cross-validation uses each training fold as validation once and reports F1 for each fold plus the mean. This is a stability check, not a model tournament or threshold-tuning exercise. The unweighted Logistic Regression and fixed 0.50 threshold are retained as the transparent baseline.

In [ ]:
model = build_delivered_order_logistic_pipeline(random_state=RANDOM_SEED)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_f1_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1")

print("5-fold F1 scores:", np.round(cv_f1_scores, 3))
print(f"Mean F1: {cv_f1_scores.mean():.3f}")

model.fit(X_train, y_train)

## 4. Evaluate on the held-out test set

The test set is used only after training. Logistic Regression outputs negative-review probabilities; probability at or above 0.50 predicts class `1`. Because negative reviews are the minority, accuracy alone is not used.

* **Precision:** proportion of predicted negative reviews that are actually negative.
* **Recall:** proportion of actual negative reviews identified.
* **F1:** harmonic mean of precision and recall.
* **Balanced Accuracy:** average recall across both classes.
* **PR AUC:** precision-recall ranking quality for the negative class.
* **ROC AUC:** probability-ranking ability across both classes.

In [ ]:
test_probability = model.predict_proba(X_test)[:, 1]
test_metrics = classification_metrics(y_test, test_probability, threshold=0.50)
display(pd.Series(test_metrics, name="test_set").round(3))

test_prediction = (test_probability >= 0.50).astype(int)
print(classification_report(
    y_test,
    test_prediction,
    target_names=["Review 3–5", "Negative review 1–2"],
    zero_division=0,
))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_prediction,
    display_labels=["Review 3–5", "Negative 1–2"],
)

## 5. Interpret the model

A positive coefficient is associated with higher odds of a negative review; a negative coefficient is associated with lower odds. `exp(coefficient)` is the odds ratio. Numeric coefficients refer to one standard deviation because numeric inputs are scaled. These are associations, not causal effects.

In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["classifier"].coef_[0]
coefficient_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
})
coefficient_table["odds_ratio"] = np.exp(coefficient_table["coefficient"])
strongest_associations = pd.concat([
    coefficient_table.nsmallest(10, "coefficient"),
    coefficient_table.nlargest(10, "coefficient"),
]).drop_duplicates()
display(
    strongest_associations
    .sort_values("coefficient")
    .reset_index(drop=True)
    .round(3)
)

## 6. Save the delivered-order model

`save_delivered_order_inference_artifact(...)` stores the fitted preprocessing, Logistic Regression, fixed threshold, delivered-order feature contract, prediction point and metadata in a distinct artifact. The smoke check reloads it and applies the saved contract to one held-out row.

In [ ]:
ARTIFACT_PATH = (
    PROJECT_ROOT / "deployment" / "delivered_order_negative_review_pipeline.pkl"
)
save_delivered_order_inference_artifact(
    model,
    str(ARTIFACT_PATH),
    threshold=0.50,
    metadata={
        "training_data_contract": "Phase 1 order_level.csv + item_level.csv",
        "population": "all delivered orders with a recorded review",
        "model": "Logistic Regression",
        "classification_threshold": 0.50,
    },
)
artifact = joblib.load(ARTIFACT_PATH)
display(predict_from_delivered_order_artifact(artifact, X_test.iloc[[0]]))
print(f"Saved model artifact: {ARTIFACT_PATH}")